# Kathmandu traffic vision - plate reader run 4

Run 4 of the plate reader: the **character detector** fine-tuned on real Kathmandu plates. Boxes come from matching AI-read plate text (training photos only) to the run-1 detector's low-confidence candidates; unmatched candidates such as bolts become background. Scored per source, then end to end on 112 AI-labelled test plates against the run-1 detector, run once.

In [ ]:
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
"""Read a number plate: plate crop -> character boxes (chars_best.pt) -> one label per box
(glyphs_best.pt) -> text in reading order. Models come from the train-plate-reader notebook.

    python reader.py plate.jpg [--chars chars_best.pt] [--glyphs glyphs_best.pt]
"""
from __future__ import annotations

import argparse
import re

import cv2

Box = tuple  # (x1, y1, x2, y2, label, conf)


def reading_order(boxes: list[Box]) -> list[Box]:
    """Return the character boxes in the order a person reads the plate.

    Nepali plates come in one row (most car plates) or two rows (most motorcycle plates, and the
    upper line of zone + lot over the number on older red plates). Read the top row left to right,
    then the bottom row left to right.
    """
    # New row where the gap between consecutive vertical centres exceeds half a character height:
    # scales with plate size, and a tilted row's small steady steps never reach it.
    heights = sorted(b[3] - b[1] for b in boxes)
    gap = heights[len(heights) // 2] / 2
    rows, last = [], None
    for b in sorted(boxes, key=lambda b: (b[1] + b[3]) / 2):
        c = (b[1] + b[3]) / 2
        if last is None or c - last > gap:
            rows.append([])
        rows[-1].append(b)
        last = c
    return [b for row in rows for b in sorted(row, key=lambda b: b[0])]


DEV_DIGITS = set("०१२३४५६७८९")
EMB_DIGITS = set("0123456789")
DEV_PLATE = re.compile(r"L+D+L+D{1,4}")   # zone (or province), lot, vehicle class, number: बा २ च १२३४

# shortcut: code lists written from memory of Nepali plate formats; confirm against the Department of
# Transport Management list before relying on the check for enforcement
ZONES = ["मे", "को", "स", "ज", "बा", "ना", "ग", "लु", "ध", "रा", "भे", "क", "से", "म"]   # 14 old zones
PROVINCES = ["को", "म", "बा", "ग", "लु", "क", "सु",                                    # abbreviations
             "कोशी", "मधेश", "बागमती", "गण्डकी", "लुम्बिनी", "कर्णाली", "सुदूरपश्चिम"]    # full names


def valid_prefix(text: str) -> bool:
    """Does a Devanagari reading start with a real zone or province code, followed by the lot digits?

    Accepted: zone + digit (बा२...), "प्रदेश" + digit (प्रदेश३...), "प्र" + digit (प्र३...),
    province + "प्रदेश" (बागमतीप्रदेश..., बाप्रदेश...), province abbreviation + "प्र" + digit (बाप्र०६...).
    """
    starts_digit = lambda rest: rest[:1] in DEV_DIGITS
    for p in ("प्रदेश", "प्र"):
        if text.startswith(p) and starts_digit(text[len(p):]):
            return True
    for p in PROVINCES:
        if text.startswith(p + "प्रदेश") or (text.startswith(p + "प्र") and starts_digit(text[len(p) + len("प्र"):])):
            return True
    return any(text.startswith(z) and starts_digit(text[len(z):]) for z in ZONES)


def check_plate(labels: list[str], conf: float, min_conf: float = 0.6) -> tuple[bool, str]:
    """Is this reading plausible enough to use without a person checking it? (ok, reason)

    Catches structure errors: missed or extra characters, a letter in a number slot, mixed scripts,
    and unsure characters. It cannot catch a look-alike swapped for another of the same kind
    (३ for ६, ना for बा): those only show up as low confidence, if at all.
    """
    if not labels:
        return False, "no characters read"
    if conf < min_conf:
        return False, f"unsure character (confidence {conf:.2f} < {min_conf})"
    dev = [lab for lab in labels if not (lab in EMB_DIGITS or lab.isascii())]
    if dev and len(dev) < len(labels):
        return False, "mixes Devanagari and embossed characters"
    if dev:
        shape = "".join("D" if lab in DEV_DIGITS else "L" for lab in labels)
        if not DEV_PLATE.fullmatch(shape):
            return False, f"not a Nepali plate layout (letters/digits: {shape})"
        if not valid_prefix("".join(labels)):
            return False, f"unknown zone or province code at the start ({''.join(labels)[:4]})"
        return True, "ok"
    # shortcut: embossed plates are only checked for digits and letters both being present;
    # add their exact layout once it is confirmed against real plates
    if not any(lab in EMB_DIGITS for lab in labels) or all(lab in EMB_DIGITS for lab in labels):
        return False, "embossed reading without both letters and digits"
    return True, "ok"


def read_plate(plate, chars, glyphs, min_conf: float = 0.55) -> tuple[list[str], float]:
    """Character labels in reading order, and the lowest confidence (a weak link makes the whole reading weak).

    min_conf 0.55: chosen on the 188 training plates (labelling/eval_reader.py); at the old 0.25 the character
    detector also boxed plate bolts and screws as characters.
    """
    boxes = []
    for x1, y1, x2, y2 in chars.predict(plate, imgsz=416, conf=min_conf, verbose=False)[0].boxes.xyxy.tolist():
        crop = plate[int(y1):int(y2), int(x1):int(x2)]
        if crop.size == 0:
            continue
        r = glyphs.predict(crop, imgsz=64, verbose=False)[0]
        label = r.names[r.probs.top1]
        if label != "flag":
            boxes.append((x1, y1, x2, y2, label.removeprefix("emb_"), float(r.probs.top1conf)))
    if not boxes:
        return [], 0.0
    ordered = reading_order(boxes)
    return [b[4] for b in ordered], min(b[5] for b in ordered)


def plate_crop(img, plates, conf: float = 0.2):
    """Crop the plate out of a photo or vehicle crop, or None. Uses the union of all plate boxes: on close-ups
    the plate detector splits one plate into pieces. Padded so edge characters are not cut off."""
    b = plates.predict(img, imgsz=1280, conf=conf, verbose=False)[0].boxes.xyxy.tolist()
    if not b:
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = min(v[0] for v in b), min(v[1] for v in b), max(v[2] for v in b), max(v[3] for v in b)
    px, py = (x2 - x1) * 0.08, (y2 - y1) * 0.15
    return img[int(max(0, y1 - py)):int(min(h, y2 + py)), int(max(0, x1 - px)):int(min(w, x2 + px))]


def read_vehicle(frame, box, plates, chars, glyphs) -> tuple[str, str, object]:
    """(plate text, "ok" or "review: reason", plate crop or None) for the vehicle inside box (x1, y1, x2, y2)."""
    x1, y1, x2, y2 = (int(v) for v in box[:4])
    crop = plate_crop(frame[max(0, y1):y2, max(0, x1):x2], plates)
    if crop is None or crop.size == 0:
        return "", "review: no plate found", None
    labels, conf = read_plate(crop, chars, glyphs)
    ok, why = check_plate(labels, conf)
    return "".join(labels), "ok" if ok else "review: " + why, crop



In [ ]:

import os, random, yaml
from pathlib import Path
IN, OUT = Path("/kaggle/input/datasets"), Path("/kaggle/working/data")
VAL_LISTS = {}

def label_of(img):
    parts = list(img.parts)
    if "images" in parts:                                       # YOLO layout: .../images/... -> .../labels/...
        i = len(parts) - 1 - parts[::-1].index("images")
        p = Path(*parts[:i], "labels", *parts[i + 1:]).with_suffix(".txt")
        if p.exists():
            return p
    return img.with_suffix(".txt")                              # some sets keep labels next to images

def add(tag, split, images, names, mapping, repeat=1):
    """Link images into OUT/<split>, rewriting labels: keep only source classes in `mapping`
    (source name -> target id); boxes of other classes are dropped, the image stays."""
    d = OUT / split
    (d / "images").mkdir(parents=True, exist_ok=True)
    (d / "labels").mkdir(parents=True, exist_ok=True)
    n_box, kept = 0, []
    for img in images:
        rows, lab = [], label_of(img)
        if lab.exists():
            for ln in lab.read_text(errors="ignore").splitlines():
                parts = ln.split()
                if len(parts) != 5 or not parts[0].isdigit():      # skip polygons / malformed rows
                    continue
                k = int(parts[0])
                if k < len(names) and names[k] in mapping:
                    rows.append(" ".join([str(mapping[names[k]])] + parts[1:]))
        for r in range(repeat):
            stem = f"{tag}_{r}_{img.parent.name}_{img.stem}".replace(" ", "_")
            link = d / "images" / (stem + img.suffix.lower())
            if not link.exists():
                link.symlink_to(img)
            (d / "labels" / (stem + ".txt")).write_text("\n".join(rows))
            kept.append(str(link))
        n_box += len(rows)
    if split == "val":
        VAL_LISTS[tag] = kept
    print(f"{tag:<10} {split:<5} {len(images):>6} images x{repeat}  {n_box:>7} boxes kept")

def imgs(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

def split_random(images, val_share=0.15, seed=0):
    images = images[:]
    random.Random(seed).shuffle(images)
    n = int(len(images) * val_share)
    return images[n:], images[:n]

def write_yaml(names):
    (OUT / "data.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "val/images",
                                                 "names": dict(enumerate(names))}))
    for tag, files in VAL_LISTS.items():          # per-source validation sets, to see where the model is weak
        (OUT / f"val_{tag}.txt").write_text("\n".join(files))
        (OUT / f"data_{tag}.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images",
                                                            "val": f"val_{tag}.txt", "names": dict(enumerate(names))}))


In [ ]:

import zipfile, re, cv2, numpy as np, pandas as pd, shutil, glob, yaml
from collections import Counter
from ultralytics import YOLO
find = lambda name: sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
plates_m = YOLO(find("plates_best.pt")[0])
old = find("chars_best.pt")[0]; chars_old = YOLO(old)
glyphs = YOLO([p for p in find("glyphs_best.pt") if "run-3" in p][0])
PH = Path("/kaggle/working/photos"); zipfile.ZipFile(find("photos.zip")[0]).extractall(PH)
read_tsv = lambda f: {l.split("\t")[0]: l.split("\t")[1] for l in open(f, encoding="utf-8").read().splitlines() if "\t" in l}
LAB = read_tsv(find("labels_claude_train.txt")[0])
MARKS = {chr(c) for c in [*range(0x0900, 0x0904), *range(0x093A, 0x094D), 0x094E, 0x094F, 0x0962, 0x0963]}

def units(text):
    out = []
    for ch in text:
        if out and (ch in MARKS or ch == "्" or out[-1].endswith("्")):
            out[-1] += ch
        else:
            out.append(ch)
    return out

def plate_crop(img):
    b = plates_m.predict(img, imgsz=1280, conf=0.2, verbose=False)[0].boxes.xyxy.tolist()
    if not b:
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = min(v[0] for v in b), min(v[1] for v in b), max(v[2] for v in b), max(v[3] for v in b)
    px, py = (x2 - x1) * 0.08, (y2 - y1) * 0.15
    return img[int(max(0, y1 - py)):int(min(h, y2 + py)), int(max(0, x1 - px)):int(min(w, x2 + px))]

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    i = ix * iy
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-9)

def align(score):
    # pick m of the n boxes, keeping reading order, maximising the summed score (n >= m)
    n, m = score.shape
    best = np.full((n + 1, m + 1), -1e9); best[:, 0] = 0
    for i in range(1, n + 1):
        for j in range(1, min(i, m) + 1):
            best[i, j] = max(best[i - 1, j], best[i - 1, j - 1] + score[i - 1, j - 1])
    pick, i, j = [], n, m
    while j:
        if i - 1 >= j and best[i, j] == best[i - 1, j]:
            i -= 1
        else:
            pick.append(i - 1); i -= 1; j -= 1
    return pick[::-1]
assert align(np.array([[0, 0], [1, 0], [0, 0], [0, 1]], float)) == [1, 3]    # skips the two "bolts"

name_id = {v: k for k, v in glyphs.names.items()}
REAL = Path("/kaggle/working/real"); (REAL / "images").mkdir(parents=True, exist_ok=True)
stats, photos = Counter(), {}
for pid, text in LAB.items():
    crop = plate_crop(cv2.imread(str(PH / f"{pid}.jpg")))
    if crop is None:
        stats["no plate found"] += 1; continue
    cand = [(x1, y1, x2, y2, "", 1.0) for x1, y1, x2, y2 in
            chars_old.predict(crop, imgsz=416, conf=0.05, verbose=False)[0].boxes.xyxy.tolist()]
    cand = [b for b in cand if b[2] - b[0] > 2 and b[3] - b[1] > 2]
    u = units(text)
    if len(cand) < len(u):
        stats["fewer candidates than characters"] += 1; continue
    cand = reading_order(cand)
    probs = [r.probs.data.cpu().numpy() for r in glyphs.predict(
        [crop[int(b[1]):int(b[3]), int(b[0]):int(b[2])] for b in cand], imgsz=64, verbose=False)]
    score = np.array([[pr[name_id[x]] if x in name_id else 0.3 for x in u] for pr in probs])
    pick = align(score)
    chosen = [cand[i] for i in pick]
    if any(iou(a, b) > 0.5 for k, a in enumerate(chosen) for b in chosen[k + 1:]):
        stats["two characters on one box"] += 1; continue
    known = [(probs[i].argmax(), name_id[x]) for i, x in zip(pick, u) if x in name_id]
    if known and sum(a == b for a, b in known) < 0.6 * len(known):
        stats["classifier disagrees"] += 1; continue
    h, w = crop.shape[:2]
    f = REAL / "images" / f"{pid}.jpg"; cv2.imwrite(str(f), crop)
    f.with_suffix(".txt").write_text("\n".join(
        f"0 {(b[0] + b[2]) / 2 / w:.6f} {(b[1] + b[3]) / 2 / h:.6f} {(b[2] - b[0]) / w:.6f} {(b[3] - b[1]) / h:.6f}"
        for b in chosen))
    photos[pid] = f; stats["used"] += 1; stats["background boxes"] += len(cand) - len(chosen)
print("real plates:", dict(stats))

ids = sorted(photos); random.Random(0).shuffle(ids); nv = max(1, len(ids) * 20 // 100)
cha = IN / "inspiring-lab/nepali-vehicles-number-plate-dataset/open_source_nepali_plates"
tr, va = split_random(imgs(cha / "images"))
add("plates_dev", "train", tr, ["character"], {"character": 0})
add("plates_dev", "val", va, ["character"], {"character": 0})
add("kathmandu_real", "train", [photos[k] for k in ids[nv:]], ["character"], {"character": 0}, repeat=4)
add("kathmandu_real", "val", [photos[k] for k in ids[:nv]], ["character"], {"character": 0})
write_yaml(["character"])

det = YOLO(old)                                  # fine-tune run 1, not from scratch
det.train(data=str(OUT / "data.yaml"), epochs=60, time=1.5, imgsz=416, batch=32, patience=15, lr0=0.002,
          project="/kaggle/working/runs", name="chars4", workers=4, seed=0, plots=True)
shutil.copy(sorted(glob.glob("/kaggle/working/runs/chars4*/weights/best.pt"))[-1], "/kaggle/working/chars_best.pt")
new = YOLO("/kaggle/working/chars_best.pt")

rows = []
for tag in ("plates_dev", "kathmandu_real"):
    y = OUT / f"data_{tag}.yaml"
    y.write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": f"val_{tag}.txt", "names": {0: "character"}}))
    for nm, m in (("run1", chars_old), ("run4", new)):
        r = m.val(data=str(y), imgsz=416, verbose=False, plots=False)
        rows.append({"val_source": tag, "model": nm, "mAP50": round(r.box.map50, 3), "precision": round(r.box.mp, 3),
                     "recall": round(r.box.mr, 3)})
print(pd.DataFrame(rows).to_string(index=False)); pd.DataFrame(rows).to_csv("/kaggle/working/chars_per_source.csv", index=False)

# End to end. The new detector's threshold is chosen on held-out TRAINING photos; the test photos are run once.
digits = lambda s: (re.findall(r"[०-९]+$", s) or [""])[0]
def score(labels_by_id, chars, conf):
    ex = num = ok = okex = 0
    for pid, truth in labels_by_id.items():
        crop = plate_crop(cv2.imread(str(PH / f"{pid}.jpg")))
        labels, cf = read_plate(crop, chars, glyphs, min_conf=conf) if crop is not None else ([], 0.0)
        got = "".join(labels); good = check_plate(labels, cf)[0]
        ex += got == truth; num += digits(got) == digits(truth); ok += good; okex += good and got == truth
    n = len(labels_by_id)
    return {"exact": round(ex / n, 3), "number": round(num / n, 3), "pass_check": round(ok / n, 3),
            "exact_among_passing": round(okex / max(1, ok), 3), "n": n}
held = {k: LAB[k] for k in ids[:nv]}
sweep = {c: score(held, new, c) for c in (0.25, 0.35, 0.45, 0.55, 0.65)}
print("threshold sweep on held-out training photos:", sweep)
pick_c = max(sweep, key=lambda c: (sweep[c]["exact"], c))
TEST = read_tsv(find("test_ai_labelled.tsv")[0])
res = pd.DataFrame([{"detector": "run1 @0.55", **score(TEST, chars_old, 0.55)},
                    {"detector": f"run4 @{pick_c}", **score(TEST, new, pick_c)}])
res.to_csv("/kaggle/working/end2end_test.csv", index=False)
print("TEST (112 AI-labelled plates, run once):"); print(res.to_string(index=False))
shutil.rmtree(OUT); shutil.rmtree(PH); shutil.rmtree(REAL)
